# 08 — Python Security Automation Scripts

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Security Automation  
**Tools:** Python 3, os, subprocess, hashlib, requests, socket

---

## Objectives
- Automate common security tasks with Python scripts
- Build a password strength checker
- Hash and verify file integrity
- Perform basic port scanning
- Automate log file monitoring for anomalies
- Generate a security audit report

## 1. Setup & Imports

In [ ]:
import os
import re
import json
import socket
import hashlib
import secrets
import string
from datetime import datetime
from pathlib import Path

## 2. Password Strength Checker

A strong password should have:
- At least **12 characters**
- **Uppercase** and **lowercase** letters
- At least one **number**
- At least one **special character**
- No common dictionary words

In [ ]:
COMMON_PASSWORDS = {'password', 'password123', '123456', 'admin', 'letmein', 'qwerty', 'welcome'}

def check_password_strength(password: str) -> dict:
    issues = []
    score  = 0

    if password.lower() in COMMON_PASSWORDS:
        return {'score': 0, 'rating': 'CRITICAL', 'issues': ['Password is in common passwords list']}

    if len(password) >= 12:   score += 2
    elif len(password) >= 8:  score += 1
    else: issues.append('Too short (minimum 12 characters)')

    if re.search(r'[A-Z]', password): score += 1
    else: issues.append('Missing uppercase letter')

    if re.search(r'[a-z]', password): score += 1
    else: issues.append('Missing lowercase letter')

    if re.search(r'\d', password):    score += 1
    else: issues.append('Missing number')

    if re.search(r'[!@#$%^&*()_+\-=\[\]{};:,.<>?]', password): score += 2
    else: issues.append('Missing special character')

    rating = {7: 'STRONG', 5: 'MODERATE', 3: 'WEAK'}
    label  = next((v for k, v in rating.items() if score >= k), 'VERY WEAK')

    return {'score': score, 'max_score': 7, 'rating': label, 'issues': issues}


# Test passwords
test_passwords = ['password123', 'Admin1!', 'Tr0ub4dor&3_secure', 'abc', 'C0mplex!Pass#2025']

for pwd in test_passwords:
    result = check_password_strength(pwd)
    print(f"  {pwd:25} -> {result['rating']:10} (score {result['score']}/7)")
    for issue in result['issues']:
        print(f"    ⚠ {issue}")

## 3. Secure Password Generator

In [ ]:
def generate_password(length: int = 16, exclude_ambiguous: bool = True) -> str:
    """Generate a cryptographically secure random password."""
    chars = string.ascii_letters + string.digits + '!@#$%^&*()_+-='
    if exclude_ambiguous:
        # Remove O, 0, l, 1, I — visually ambiguous
        chars = ''.join(c for c in chars if c not in 'O0lI1')
    # Guarantee at least one of each required type
    password = [
        secrets.choice(string.ascii_uppercase.replace('O','').replace('I','')),
        secrets.choice(string.ascii_lowercase.replace('l','')),
        secrets.choice(string.digits.replace('0','').replace('1','')),
        secrets.choice('!@#$%^&*()_+-=')
    ]
    password += [secrets.choice(chars) for _ in range(length - 4)]
    secrets.SystemRandom().shuffle(password)
    return ''.join(password)


print('Generated secure passwords:')
for _ in range(5):
    pwd = generate_password(16)
    result = check_password_strength(pwd)
    print(f"  {pwd}  -> {result['rating']}")

## 4. File Integrity Checker (Hashing)

File integrity monitoring detects unauthorized changes to critical files.
The process:
1. **Baseline** — hash all files and store results
2. **Monitor** — re-hash periodically and compare
3. **Alert** — flag any file whose hash changed

In [ ]:
def hash_file(filepath: str, algorithm: str = 'sha256') -> str:
    """Return hex digest of a file using the given hash algorithm."""
    h = hashlib.new(algorithm)
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


def create_baseline(directory: str, pattern: str = '*.py') -> dict:
    """Create a hash baseline for all matching files in a directory."""
    baseline = {}
    for path in Path(directory).rglob(pattern):
        try:
            baseline[str(path)] = hash_file(str(path))
        except (PermissionError, FileNotFoundError):
            pass
    return baseline


def check_integrity(baseline: dict) -> list:
    """Compare current file hashes against baseline. Return list of alerts."""
    alerts = []
    for filepath, original_hash in baseline.items():
        if not os.path.exists(filepath):
            alerts.append({'type': 'DELETED',  'file': filepath})
        else:
            current_hash = hash_file(filepath)
            if current_hash != original_hash:
                alerts.append({'type': 'MODIFIED', 'file': filepath})
    return alerts


# Demo — hash this notebook's own directory
print('Supported hash algorithms:', ['md5', 'sha1', 'sha256', 'sha512'])
print()

# Demo with string content (simulating file hash)
demo_content = b'Critical system file content v1.0'
for algo in ['md5', 'sha1', 'sha256']:
    h = hashlib.new(algo, demo_content).hexdigest()
    print(f'  {algo:8}: {h}')

## 5. Basic Port Scanner

In [ ]:
def scan_ports(host: str, ports: list, timeout: float = 0.5) -> dict:
    """Scan a list of TCP ports on a host. Returns open/closed results."""
    results = {'host': host, 'open': [], 'closed': [], 'timestamp': datetime.now().isoformat()}

    WELL_KNOWN = {
        21: 'FTP', 22: 'SSH', 23: 'Telnet', 25: 'SMTP',
        53: 'DNS', 80: 'HTTP', 110: 'POP3', 143: 'IMAP',
        443: 'HTTPS', 445: 'SMB', 3306: 'MySQL',
        3389: 'RDP', 5432: 'PostgreSQL', 8080: 'HTTP-Alt'
    }

    for port in ports:
        try:
            with socket.create_connection((host, port), timeout=timeout):
                service = WELL_KNOWN.get(port, 'unknown')
                results['open'].append({'port': port, 'service': service})
        except (socket.timeout, ConnectionRefusedError, OSError):
            results['closed'].append(port)

    return results


# Scan localhost — safe to run anywhere
common_ports = [22, 80, 443, 3306, 3389, 8080, 5432]
result = scan_ports('127.0.0.1', common_ports, timeout=0.3)

print(f"Port scan results for {result['host']}")
print(f"Scanned at: {result['timestamp']}\n")

if result['open']:
    print('OPEN PORTS:')
    for p in result['open']:
        print(f"  ✓ Port {p['port']:5} — {p['service']}")
else:
    print('No open ports found on localhost.')

print(f"\nClosed/filtered: {result['closed']}")

## 6. Automated Log Monitor

This script watches a log file for new suspicious lines in real time.  
Here we simulate it by processing a batch of new entries.

In [ ]:
# Threat signatures to watch for in any log
THREAT_SIGNATURES = [
    (re.compile(r'Failed password',          re.I), 'MEDIUM',   'SSH_FAIL'),
    (re.compile(r'Invalid user',             re.I), 'MEDIUM',   'INVALID_USER'),
    (re.compile(r'sudo.*FAILED',             re.I), 'HIGH',     'SUDO_FAIL'),
    (re.compile(r'segfault|kernel panic',    re.I), 'HIGH',     'SYSTEM_ERROR'),
    (re.compile(r'sql injection|<script>',   re.I), 'CRITICAL', 'INJECTION_ATTEMPT'),
    (re.compile(r'\.\./\.\./|etc/passwd',    re.I), 'CRITICAL', 'PATH_TRAVERSAL'),
]

def monitor_log_lines(lines: list) -> list:
    """Scan log lines against threat signatures and return matches."""
    findings = []
    for i, line in enumerate(lines, 1):
        for pattern, severity, rule_id in THREAT_SIGNATURES:
            if pattern.search(line):
                findings.append({
                    'line_no'  : i,
                    'severity' : severity,
                    'rule'     : rule_id,
                    'log_entry': line.strip()
                })
    return findings


# Simulated incoming log lines
new_log_lines = [
    'Jan 16 07:01:03 server sshd: Failed password for root from 45.33.32.156',
    'Jan 16 07:01:05 server sshd: Invalid user deploy from 45.33.32.156',
    'Jan 16 07:05:00 server sudo: bency : FAILED ; TTY=pts/0 ; PWD=/home/bency',
    'Jan 16 07:10:12 server apache: GET /../../../etc/passwd HTTP/1.1 400',
    'Jan 16 07:15:00 server app: User logged in successfully',
    'Jan 16 07:20:01 server app: GET /search?q=<script>alert(1)</script> 400',
]

findings = monitor_log_lines(new_log_lines)
print(f'Scanned {len(new_log_lines)} lines — {len(findings)} threat(s) detected:\n')
for f in findings:
    print(f"  [{f['severity']:8}] Rule: {f['rule']:20} Line {f['line_no']}: {f['log_entry'][:70]}")

## 7. Security Audit Report Generator

In [ ]:
from collections import Counter

audit_report = {
    'report_generated' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'host'             : socket.gethostname(),
    'modules_run'      : [
        'password_strength_checker',
        'file_integrity_checker',
        'port_scanner',
        'log_monitor'
    ],
    'port_scan': {
        'target'      : result['host'],
        'open_ports'  : result['open'],
        'closed_ports': len(result['closed'])
    },
    'log_monitor': {
        'lines_scanned'    : len(new_log_lines),
        'threats_detected' : len(findings),
        'by_severity'      : dict(Counter(f['severity'] for f in findings)),
        'findings'         : findings
    },
    'recommendations': [
        'Enforce SSH key-based authentication; disable password login',
        'Implement fail2ban or equivalent to auto-block brute force IPs',
        'Enable a Web Application Firewall (WAF) to block injection attempts',
        'Schedule daily file integrity baseline checks',
        'Review all CRITICAL and HIGH findings within 1 hour of detection'
    ]
}

print(json.dumps(audit_report, indent=2))

## 8. Key Takeaways

| Script | Security Purpose |
|--------|----------------|
| Password checker | Enforce strong credential policies |
| Password generator | Remove human bias from password creation |
| File integrity checker | Detect tampering with critical files (FIM) |
| Port scanner | Inventory exposed services; find misconfigurations |
| Log monitor | Real-time threat detection without a full SIEM |
| Audit report | Document security posture for compliance/review |

- Python's `secrets` module is **cryptographically secure** — always prefer it over `random` for security use cases
- `hashlib` with `sha256` is the minimum standard for integrity checks (avoid `md5`/`sha1` in production)
- Automation scripts reduce response time from minutes to seconds

---
*Next: 09 — Network Reconnaissance & Scanning*